In [2]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io
import traceback 
import ast 

import numpy as np
import matplotlib
#matplotlib.use('TkAgg')
import matplotlib.pyplot as plt

plt.ion()
#get_ipython().run_line_magic('matplotlib', 'inline')
# some plots
#get_ipython().run_line_magic('matplotlib', 'qt')
# the interactive plot
from matplotlib.patches import Circle
from matplotlib.colors import LogNorm
from matplotlib.colors import SymLogNorm
from matplotlib.path import Path
from skimage.measure import find_contours
from ast import literal_eval

#import astropy.units as u
from astropy.io import fits
from astropy import wcs
#from astropy.wcs import WCS
from astropy.io import ascii
#from astropy.coordinates import SkyCoord
#from astropy.coordinates import ICRS, Galactic, FK4, FK5
from astropy.visualization import make_lupton_rgb
#from astropy.modeling import models, fitting
from scipy import interpolate
#import itertools
import sys
import math
import csv
#import pylab as py
import copy
import os
import pandas as pd
from astropy.nddata import Cutout2D

# These lines supress warnings
import warnings
warnings.filterwarnings('ignore')

In [3]:
import io
import pandas as pd
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Authenticate
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

flow = InstalledAppFlow.from_client_secrets_file(
    'client_secret_527671004811-mov0s2s96a9gqsdim7h0qtl3od2dnuos.apps.googleusercontent.com.json',
    SCOPES
)

creds = flow.run_local_server(port=0)
service = build('drive', 'v3', credentials=creds)

# Folder ID
folder_id = "1E9w5Irzj0FP9VUbCVYHM3PQtuMmLZtJU2FQTkaqCcNnv_ZANHuTlFqbayGdb0v9jZYQQwz3C"

# Only return files whose MIME type is text/csv
query = (
    f"'{folder_id}' in parents "
    f"and mimeType='text/csv'"
)

results = service.files().list(
    q=query,
    fields="files(id,name)"
).execute()

files = results.get("files", [])

print(f"Found {len(files)} CSV files")

all_data = []

for file in files:

    file_id = file["id"]
    filename = file["name"]

    print(f"Loading: {filename}")

    request = service.files().get_media(fileId=file_id)

    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)

    done = False
    while not done:
        status, done = downloader.next_chunk()
        if status:
            print(f"{status.progress() * 100:.1f}%")

    fh.seek(0)

    df = pd.read_csv(fh, header=0, skiprows=[1])

    # Remove units row if present
    if len(df) > 0 and str(df.iloc[0]["YB"]) == "ID Number":
        df = df.iloc[1:].reset_index(drop=True)

    all_data.append(df)

print("\nDONE LOADING ALL CSV FILES")

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=527671004811-mov0s2s96a9gqsdim7h0qtl3od2dnuos.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A49510%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=ri6ihTjsm5OreDtYbgIeU0bGQEZmFR&code_challenge=QgbtgSCxmkHGzbC51qAji8yjUieL4GU87xzrBQVbgIg&code_challenge_method=S256&access_type=offline
Found 27 CSV files
Loading: YBphotometry_results_DBWU - Jeff Hoffman.csv
100.0%
Loading: YBphotometry_results_colemaya - Michael Coleman.csv
100.0%
Loading: YBphotometry_results_Qu_handran_may24 - Caden Handran.csv
100.0%
Loading: YBphotometry_results_Qu_Morgan_May24 - samaje morgan.csv
100.0%
Loading: YBphotometry_results_Qu_Miller_may24 - Whitt Miller.csv
100.0%
Loading: YBphotometry_results_Qu_Fieuw_may24 - Brielle Fieuw.csv
100.0%
Loading: YBphotometry_results_Qu_Qu_may24 - Dawid.csv
100.0%
Loading: YBphotometry_results_Qu_Stowman_May24 - I

In [4]:

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

flow = InstalledAppFlow.from_client_secrets_file(
    'client_secret_527671004811-mov0s2s96a9gqsdim7h0qtl3od2dnuos.apps.googleusercontent.com.json',
    SCOPES
)

creds = flow.run_local_server(port=0)

service = build('drive', 'v3', credentials=creds)

# File ID of the CSV
file_id = "1aNrUvi-GaKBhbK3k0QWrJzryypa2s34FQDtr3cuu9OY"

# Download the file into memory
request = service.files().export_media(
    fileId=file_id,
    mimeType="text/csv"
)
fh = io.BytesIO()

downloader = MediaIoBaseDownload(fh, request)

done = False
while not done:
    status, done = downloader.next_chunk()
    if status:
        print(f"{status.progress() * 100:.1f}%")

# Read with pandas
fh.seek(0)
control = pd.read_csv(fh, skiprows=[1])

print(control.head())

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=527671004811-mov0s2s96a9gqsdim7h0qtl3od2dnuos.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A49531%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=y8zanIBfBegEja4MrJKySVEQ6GLuoH&code_challenge=UdtLobgCnjPiTDdu56B_BxpR-MaQrj3lU09ACx-XKLo&code_challenge_method=S256&access_type=offline
100.0%
   YB   YB_long    YB_lat vertices 8 vertices 12 vertices 24 vertices 70  \
0   1 -0.041433  0.167050        NaN         NaN         NaN         NaN   
1   2 -0.023111  0.169160        NaN         NaN         NaN         NaN   
2   3  0.040876  0.020201        NaN         NaN         NaN         NaN   
3   4  0.186037 -0.611814        NaN         NaN         NaN         NaN   
4   5  0.200751 -0.513361        NaN         NaN         NaN         NaN   

   8umphotom  8flag1  8flag2  ...  24flag6  24flag7  24flag8  70umphotom  \
0        N

In [5]:
def polygon_to_mask(vertices, shape=(100, 100)):
    y, x = np.mgrid[:shape[0], :shape[1]]
    points = np.column_stack((x.ravel(), y.ravel()))
    path = Path(vertices)
    mask = path.contains_points(points)
    return mask.reshape(shape)



In [6]:
control2 = all_data[0]
np.savetxt(f"/Users/wadevining/YellowBall/YB_CSV_Sorting/MasterTable1.csv", control2, fmt="%s")    



In [57]:
yb_ids = control["YB"].dropna().unique()
all_rows = []
yb_head = control.head().columns.tolist()
yb_head2 = control2.head().columns.tolist()
lengths = ["vertices 8","vertices 12","vertices 24","vertices 70"]
umlst = ["8umphotom","12umphotom","24umphotom","70umphotom"]
flag_cols = ["8flag1","8flag2","8flag3","8flag4","8flag5","8flag6","8flag7","8flag8", '12flag1', '12flag2', '12flag4', '12flag6', '12flag7', '12flag8', '24flag1', '24flag2', '24flag4', '24flag6', '24flag7', '24flag8', '70flag1', '70flag2', '70flag4', '70flag6', '70flag7', '70flag8']
old_flag_cols = ["8flag1","8flag2","8flag3","8flag4","8flag5","8flag6","8flag7","8flag8", '12flag1', '12flag2', '12flag4', '12flag6', '12flag7', '12flag8', '24flag1', '24flag2', '24flag4', '24flag6', '24flag7', '24flag8']

for yb in yb_ids:
    um8_tot = um12_tot = um24_tot = um70_tot = 0
    um8ave = um12ave = um24ave = um70ave = 0
    count = 0
    count1 = 0
    avgs_vertices = []
    for csv in all_data:
        row = csv[csv["YB"] == yb].iloc[0]
        if row["12umphotom"] == "Saturated" or pd.isna(row["12umphotom"]):
                continue
        um8_tot += float(row["8umphotom"])
        um12_tot += float(row["12umphotom"])
        um24_tot += float(row["24umphotom"])
        if list(row.index) == control.columns.tolist():
            um70_tot += float(row["70umphotom"])
        count += 1
    for length in lengths:
        shape_lst = []
        masks= []
        for csv in all_data:
            if length not in csv.columns:
                continue
           # row = csv.iloc[yb-1]
            row = csv[csv["YB"] == yb].iloc[0]
            if row["12umphotom"] == "Saturated":
                continue
           # if row[length]
            verts_str = row[length]
            
            if pd.isna(verts_str):
                continue
            try:
                verts = np.array(literal_eval(verts_str))
                shape_lst.append(verts)
            except:
                continue
        if len(shape_lst) == 0:
            continue

        shape=(100,100)

        for verti in shape_lst:
            mask = polygon_to_mask(verti,shape)
            masks.append(mask)

        masks = np.array(masks)
            
        test = masks.mean(axis=0)
        average_mask = test >= .5

        fig, ax = plt.subplots()
        cs = ax.contour(average_mask.astype(float), levels=[.5])
        plt.close(fig)

        if len(cs.allsegs[0]) > 0:
            avg_vertices = cs.allsegs[0][0]
            avg_vertices_str = str(tuple((float(x), float(y))for x, y in avg_vertices))
            avgs_vertices.append(avg_vertices_str)
            count1 = 1

    if count > 0:
        um8ave = round((um8_tot / count), 4)
        um12ave = round((um12_tot / count), 4)
        um24ave = round((um24_tot / count), 4)
        if um70_tot != 0:
            um70ave = round((um70_tot / count), 4)
    else:
        um8ave = um12ave = um24ave = um70ave = 0

    for head in yb_head[3:]:
        if head in flag_cols:
            avgs_vertices.append("")
        elif count > 0:
            if head == "8umphotom":
                avgs_vertices.append(str(um8ave))
            elif head == "12umphotom":
                avgs_vertices.append(str(um12ave))
            elif head == "24umphotom":
                avgs_vertices.append(str(um24ave))
            elif head == "70umphotom":
                if um70ave != 0:
                    avgs_vertices.append(str(um70ave))
                else:
                    avgs_vertices.append("")
        if head in lengths[:3] and count1 == 0:
            avgs_vertices.append("")
        elif head == "vertices 70":
            if len(avgs_vertices) == 3:
                avgs_vertices.append("")
            else:
                continue

    print(len(avgs_vertices))    
    big_row = [int(yb),float(control['YB_long'].iloc[int(yb)-1]),float(control['YB_lat'].iloc[int(yb)-1])]+avgs_vertices           
    all_rows.append(big_row)
   

connect = pd.DataFrame(all_rows, columns = yb_head)
connect.to_csv(f"/Users/wadevining/YellowBall/YB_CSV_Sorting/MasterTable.csv", index=False)    
    

30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30
30


KeyboardInterrupt: 